In [10]:
import duckdb
con = duckdb.connect("../dunnhumby.duckdb")
con.sql("SELECT * FROM '../hh_demographic.csv' LIMIT 5").show()

┌──────────┬─────────────────────┬─────────────┬────────────────┬──────────────────┬─────────────────────┬───────────────────┬───────────────┐
│ AGE_DESC │ MARITAL_STATUS_CODE │ INCOME_DESC │ HOMEOWNER_DESC │   HH_COMP_DESC   │ HOUSEHOLD_SIZE_DESC │ KID_CATEGORY_DESC │ household_key │
│ varchar  │       varchar       │   varchar   │    varchar     │     varchar      │       varchar       │      varchar      │     int64     │
├──────────┼─────────────────────┼─────────────┼────────────────┼──────────────────┼─────────────────────┼───────────────────┼───────────────┤
│ 65+      │ A                   │ 35-49K      │ Homeowner      │ 2 Adults No Kids │ 2                   │ None/Unknown      │             1 │
│ 45-54    │ A                   │ 50-74K      │ Homeowner      │ 2 Adults No Kids │ 2                   │ None/Unknown      │             7 │
│ 25-34    │ U                   │ 25-34K      │ Unknown        │ 2 Adults Kids    │ 3                   │ 1                 │             8 │

In [11]:
con.sql("""
    CREATE OR REPLACE TABLE demographics AS
    SELECT * FROM read_csv_auto('../hh_demographic.csv')
""")
con.sql("SHOW TABLES").show()

┌─────────────────────┐
│        name         │
│       varchar       │
├─────────────────────┤
│ campaign_households │
│ campaigns           │
│ coupons             │
│ demographics        │
│ products            │
│ redemptions         │
│ transactions        │
└─────────────────────┘



In [12]:
files = {
    "transactions": "transaction_data.csv",
    "products": "product.csv",
    "demographics": "hh_demographic.csv",
    "campaigns": "campaign_desc.csv",
    "campaign_households": "campaign_table.csv",
    "coupons": "coupon.csv",
    "redemptions": "coupon_redempt.csv",
}

for name, file in files.items():
    con.sql(f"""
        CREATE OR REPLACE TABLE {name} AS
        SELECT * FROM read_csv_auto('../{file}')
    """)

con.sql("SHOW TABLES").show()

┌─────────────────────┐
│        name         │
│       varchar       │
├─────────────────────┤
│ campaign_households │
│ campaigns           │
│ coupons             │
│ demographics        │
│ products            │
│ redemptions         │
│ transactions        │
└─────────────────────┘



In [13]:
con.sql("SELECT COUNT(*) FROM transactions").show()

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│      2595732 │
└──────────────┘



In [14]:
con.sql("SELECT SUM(SALES_VALUE) FROM transactions").show()

┌───────────────────┐
│ sum(SALES_VALUE)  │
│      double       │
├───────────────────┤
│ 8057463.079993598 │
└───────────────────┘



In [15]:
con.sql("""
    SELECT STORE_ID, SUM(SALES_VALUE) AS total_sales
    FROM transactions
    GROUP BY STORE_ID
    ORDER BY total_sales DESC
    LIMIT 5
""").show()

┌──────────┬────────────────────┐
│ STORE_ID │    total_sales     │
│  int64   │       double       │
├──────────┼────────────────────┤
│      367 │ 267614.34000000317 │
│      406 │ 216781.77000000438 │
│      361 │ 144698.85000000175 │
│      429 │ 142391.01000000187 │
│      343 │ 140455.75000000207 │
└──────────┴────────────────────┘



In [19]:
con.sql("""
    SELECT p.DEPARTMENT, ROUND(SUM(t.SALES_VALUE), 2) AS total_sales
    FROM transactions t
    JOIN products p ON t.PRODUCT_ID = p.PRODUCT_ID
    GROUP BY p.DEPARTMENT
    ORDER BY total_sales DESC
    LIMIT 5
""").show()

┌────────────┬─────────────┐
│ DEPARTMENT │ total_sales │
│  varchar   │   double    │
├────────────┼─────────────┤
│ GROCERY    │  4093814.14 │
│ DRUG GM    │  1055358.03 │
│ PRODUCE    │   557452.11 │
│ MEAT       │   548786.81 │
│ KIOSK-GAS  │   544222.28 │
└────────────┴─────────────┘

